***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 6-Optimization theory and algorithms   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Feb 11, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# FOR e-BOOK ONLY
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import mmids
seed = 535
rng = np.random.default_rng(seed)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

$\newcommand{\bSigma}{\boldsymbol{\Sigma}}$ $\newcommand{\bmu}{\boldsymbol{\mu}}$

## More advanced material: proof of Second-Order Sufficient Condition

In this section, we cover some more advanced material.

### Additional proofs

**Proof of Second-Order Sufficient Condition** We prove the *Second-Order Sufficient Condition*.

*Proof idea:* We use *Taylor's Theorem* again. This time we use the positive definiteness of the Hessian to bound the value of the function from below.

We will need a simple lemma.

**LEMMA** Let $A = (a_{i,j})_{i,j}$ and $B = (b_{i,j})_{i,j}$ be matrices in $\mathbb{R}^{n \times m}$. For any unit vectors $\mathbf{u} \in \mathbb{R}^n$ and $\mathbf{v} \in \mathbb{R}^m$

$$
\left| 
\mathbf{u}^T A \,\mathbf{v}
- \mathbf{u}^T B \,\mathbf{v}
\right|
\leq
\|A - B\|_F.
$$

$\flat$

*Proof:* By *Cauchy-Schwarz*,

\begin{align*}
\left| 
\mathbf{u}^T A \,\mathbf{v}
- \mathbf{u}^T B \,\mathbf{v}
\right|
&=
\left|
\sum_{i=1}^n \sum_{j=1}^m u_i v_j (a_{i,j} - b_{i,j})
\right|\\
&\leq \sqrt{\sum_{i=1}^n \sum_{j=1}^m u_i^2 v_j^2}
\sqrt{\sum_{i=1}^n \sum_{j=1}^m (a_{i,j} - b_{i,j})^2}\\
&= \|A - B\|_F,
\end{align*}

where we used that $\mathbf{u}$ and $\mathbf{v}$ have unit norm on the last line. $\square$

*Proof:* By *Taylor's Theorem*, for all unit vectors $\mathbf{v} \in \mathbb{R}^d$ and $\alpha \in \mathbb{R}$, there is $\xi_{\alpha} \in (0,1)$ such that

\begin{align*}
f(\mathbf{x}_0 + \alpha \mathbf{v})
&= f(\mathbf{x}_0) 
+ \nabla f(\mathbf{x}_0)^T(\alpha \mathbf{v})
+ \frac{1}{2}(\alpha \mathbf{v})^T \mathbf{H}_f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v}) (\alpha \mathbf{v})\\
&= f(\mathbf{x}_0) 
+ \frac{1}{2}\alpha^2 \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v})\,\mathbf{v}.
\end{align*}


where we used that $\nabla f(\mathbf{x}_0) = \mathbf{0}$. The second term on the last line is $0$ at $\mathbf{v} = \mathbf{0}$. Our goal is to show that it is strictly positive (except at $\mathbf{0}$) in a neighborhood of $\mathbf{0}$.

The set $\mathbb{S}^{d-1}$ of unit vectors in $\mathbb{R}^d$ is closed and bounded. The expression $\mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0) \,\mathbf{v}$, viewed as a function of $\mathbf{x}$ is continuous since it is a polynomial. Hence, by the *Extreme Value Theorem*, it attains its minimum on $\mathbb{S}^{d-1}$. By our assumption that $\mathbf{H}_f(\mathbf{x}_0)$ is positive definite, that minimum must be strictly positive, say $\mu > 0$.  

By the previous lemma (ignoring the absolute value),

$$
\mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0) \,\mathbf{v}
- \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0 + \mathbf{w})\,\mathbf{v}
\leq 
\|\mathbf{H}_f(\mathbf{x}_0)
- \mathbf{H}_f(\mathbf{x}_0  + \mathbf{w})\|_F.
$$

The Frobenius norm above is continuous in $\mathbf{w}$ as a composition of continuous functions. Moreover, we of course have at $\mathbf{w} = \mathbf{0}$ that this Frobenius norm is $0$. Hence, by definition of continuity, for any $\epsilon > 0$ -- say $\epsilon := \mu/2$ -- there is $\delta > 0$ such that $\mathbf{w} \in B_{\delta}(\mathbf{0})$ implies $\|\mathbf{H}_f(\mathbf{x}_0) - \mathbf{H}_f(\mathbf{x}_0  + \mathbf{w})\|_F < \epsilon = \mu/2$. 

Since $\mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0) \,\mathbf{v} > \mu$, the inequality in the previous display implies that

$$
\mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0 + \mathbf{w})\,\mathbf{v}
\geq \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0) \,\mathbf{v} - \|\mathbf{H}_f(\mathbf{x}_0)
- \mathbf{H}_f(\mathbf{x}_0 + \mathbf{w})\|_F
> \frac{\mu}{2}.
$$

This holds for any unit vector $\mathbf{v}$ and any $\mathbf{w} \in B_{\delta}(\mathbf{0})$.

Going back to our Taylor expansion, for $\alpha > 0$ small enough (not depending on $\mathbf{v}$), it holds that $\mathbf{w} = \xi_\alpha \alpha \mathbf{v} \in B_{\delta}(\mathbf{0})$ so that we get from the previous inequality

\begin{align*}
f(\mathbf{x}_0 + \alpha \mathbf{v})
&= f(\mathbf{x}_0) 
+ \frac{1}{2}\alpha^2 \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v})\,\mathbf{v}\\
&> f(\mathbf{x}_0) 
+ \frac{1}{4} \alpha^2 \mu \\
&> f(\mathbf{x}_0).
\end{align*}

Therefore $\mathbf{x}_0$ is a strict local minimizer. $\square$